# 人体动作识别 — ANN (全连接网络)

使用 UCI HAR Dataset 的 561 维手工提取特征，通过全连接网络识别 6 种日常动作。

## 1. 导入依赖

In [1]:
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.utils import to_categorical

## 2. 数据加载

直接读取 UCI HAR Dataset 中已提取好的 561 维特征（`X_train.txt` / `X_test.txt`），并从训练集中划分 20% 作为验证集。

In [2]:
def load_data():
    print("正在加载 561维 特征数据...")

    # 直接读取 X_train.txt (已经包含了提取好的特征，不是原始波形)
    # sep='\s+' 用于处理不定长的空格分隔
    x_train = pd.read_csv('UCI HAR Dataset/train/X_train.txt', header=None, sep='\s+').values
    y_train = pd.read_csv('UCI HAR Dataset/train/y_train.txt', header=None, sep='\s+').values

    x_test  = pd.read_csv('UCI HAR Dataset/test/X_test.txt',   header=None, sep='\s+').values
    y_test  = pd.read_csv('UCI HAR Dataset/test/y_test.txt',   header=None, sep='\s+').values

    # 从原训练集中划分验证集（20%）
    n_train = x_train.shape[0]
    indices = np.random.permutation(n_train)
    val_size = int(n_train * 0.2)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    x_val = x_train[val_indices]
    y_val = y_train[val_indices]
    x_train = x_train[train_indices]
    y_train = y_train[train_indices]

    # 标签处理: 1-6 转 0-5，再转 One-Hot
    y_train = to_categorical(y_train - 1)
    y_val   = to_categorical(y_val - 1)
    y_test  = to_categorical(y_test - 1)

    print(f"加载完成: 训练集 {x_train.shape}, 验证集 {x_val.shape}, 测试集 {x_test.shape}")
    return x_train, y_train, x_val, y_val, x_test, y_test

In [3]:
x_train, y_train, x_val, y_val, x_test, y_test = load_data()

正在加载 561维 特征数据...
加载完成: 训练集 (5882, 561), 验证集 (1470, 561), 测试集 (2947, 561)


## 3. 模型结构 — ANN

两层全连接网络：

```
Dense(128, relu) → Dropout(0.5)
  → Dense(64, relu) → Dropout(0.5)
  → Softmax(6)
```

In [4]:
def create_model(input_dim, n_classes):
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ])
    return model

model = create_model(input_dim=x_train.shape[1], n_classes=y_train.shape[1])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               71936     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 6)                 390       
                                                                 
Total params: 80,582
Trainable params: 80,582
Non-trainable params: 0
_________________________________________________________________


## 4. 训练与评估

In [6]:
model.fit(x_train, y_train, epochs=50, batch_size=64, verbose=1, validation_data=(x_val, y_val))

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'测试集准确率: {accuracy * 100:.2f}%')

Epoch 1/50
92/92 [==============================] - 0s 4ms/step - loss: 0.1140 - accuracy: 0.9595 - val_loss: 0.0514 - val_accuracy: 0.9789
Epoch 2/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1088 - accuracy: 0.9609 - val_loss: 0.0731 - val_accuracy: 0.9748
Epoch 3/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1082 - accuracy: 0.9573 - val_loss: 0.0497 - val_accuracy: 0.9810
Epoch 4/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1012 - accuracy: 0.9629 - val_loss: 0.0631 - val_accuracy: 0.9762
Epoch 5/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1156 - accuracy: 0.9558 - val_loss: 0.0854 - val_accuracy: 0.9660
Epoch 6/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1071 - accuracy: 0.9636 - val_loss: 0.0638 - val_accuracy: 0.9728
Epoch 7/50
92/92 [==============================] - 0s 2ms/step - loss: 0.1093 - accuracy: 0.9645 - val_loss: 0.0845 - val_accuracy: 0.9755
Epoch 8/50
92/92 [==